<a href="https://colab.research.google.com/github/Alynn-P/nathamon_portfolio/blob/main/weather_data_for_close_loop_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import requests
import os
csv_filename = '/content/drive/MyDrive/Colab Notebooks/weather_udaicharoen_Jan 2023 - Oct 2026.csv'
if os.path.exists(csv_filename):
  df = pd.read_csv(csv_filename)
  df = df[['datetime','precip','temp','windspeed','solarradiation','humidity']]
  df['datetime'] = pd.to_datetime(df['datetime'])
  print(f"success with csv {csv_filename}")
else:
  url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/Udai%20Charoen%2C%20Kuan%20Kalong%2C%20Satun%2C%20Thailand/2025-11-01/2026-10-31?unitGroup=us&include=days&key=Q73BLA7EDAEHG8V2VQV8MHD7E&contentType=json"
  respond = requests.get(url)
  if respond.status_code == 200:
    data = respond.json()
    days_data = data['days']
    df = pd.DataFrame(days_data)
    df = df[['datetime','precip','temp','windspeed','solarradiation','humidity']]
    print(f"success with API {len(df)} days")
  else:
    print(f"couldn't connect API: {respond.status_code}")
if 'df' in locals() and not df.empty:
    display(df.head())

success with csv /content/drive/MyDrive/Colab Notebooks/weather_udaicharoen_Jan 2023 - Oct 2026.csv


,datetime,precip,temp,windspeed,solarradiation,humidity
0,2023-01-01,0.000,79.8,12.5,233.6,75.2
1,2023-01-02,0.000,80.0,10.5,215.9,76.9
2,2023-01-03,0.000,81.1,12.8,204.9,79.3
3,2023-01-04,0.094,79.4,8.7,177.0,86.1
4,2023-01-05,0.000,81.0,11.6,204.4,80.6


In [3]:
if 'df' in locals() and not df.empty:
  df_unit = df[['datetime','precip','temp','windspeed','solarradiation','humidity']].rename(columns={
        'datetime' : 'date',
        'precip' : 'rainfall_mm',
        'temp' : 'temp_c(norm)',
        'windspeed' : 'windspeed_meter(norm)',
        'solarradiation' : 'solarradiation_mj(norm)',
        'humidity' : 'humidity_int'
  })
  df_unit['date'] = pd.to_datetime(df_unit['date'])
  df_unit['rainfall_mm'] = df_unit['rainfall_mm']*25.4
  df_unit['temp_c(norm)'] = (5/9)*(df_unit['temp_c(norm)']-32)*(1/40)
  df_unit['windspeed_meter(norm)'] = df_unit['windspeed_meter(norm)']*0.44074/10
  df_unit['solarradiation_mj(norm)'] = df_unit['solarradiation_mj(norm)']*0.0864/800
  df_unit['humidity_int'] = df_unit['humidity_int']/100
  display(df_unit.head())
else:
  print("need to check cell 1")

,date,rainfall_mm,temp_c(norm),windspeed_meter(norm),solarradiation_mj(norm),humidity_int
0,2023-01-01,0.0000,0.663889,0.550925,0.025229,0.752
1,2023-01-02,0.0000,0.666667,0.462777,0.023317,0.769
2,2023-01-03,0.0000,0.681944,0.564147,0.022129,0.793
3,2023-01-04,2.3876,0.658333,0.383444,0.019116,0.861
4,2023-01-05,0.0000,0.680556,0.511258,0.022075,0.806


In [4]:
#Last 2 days
df_unit['precip_d-1'] = df_unit['rainfall_mm'].shift(1)
df_unit['precip_d-2'] = df_unit['rainfall_mm'].shift(2)
df_unit['last_2_days'] = df_unit['precip_d-2']+df_unit['precip_d-1']
#Last 3 days
df_unit['precip_d-1'] = df_unit['rainfall_mm'].shift(1)
df_unit['precip_d-2'] = df_unit['rainfall_mm'].shift(2)
df_unit['precip_d-3'] = df_unit['rainfall_mm'].shift(3)
df_unit['last_3_days'] = df_unit['precip_d-3']+df_unit['precip_d-2']+df_unit['precip_d-1']
#Next 2 days
df_unit['precip_d+1'] = df_unit['rainfall_mm'].shift(-1)
df_unit['precip_d+2'] = df_unit['rainfall_mm'].shift(-2)
df_unit['next_2_days'] = df_unit['precip_d+2']+df_unit['precip_d+1']
#ET proxy
df_unit['ET'] = (0.5*df_unit['temp_c(norm)'])+(0.3*df_unit['windspeed_meter(norm)'])+(0.2*df_unit['solarradiation_mj(norm)'])-(0.4*df_unit['humidity_int'])
#soilfactor
df_unit['soilfactor'] = 0.8
#SMI daily
df_unit['SMI_daily'] = df_unit['rainfall_mm']-(df_unit['ET']*df_unit['soilfactor'])
#Sum 7 dats SMI
df_unit['sum_7_days_SMI'] = df_unit['SMI_daily'].rolling(window=7,min_periods=1).sum()

display(df_unit[['date','rainfall_mm','ET','SMI_daily','sum_7_days_SMI','last_2_days','last_3_days','next_2_days']].head(30))

,date,rainfall_mm,ET,SMI_daily,sum_7_days_SMI,last_2_days,last_3_days,next_2_days
0,2023-01-01,0.0000,0.201468,-0.161174,-0.161174,NaN,NaN,0.0000
1,2023-01-02,0.0000,0.169230,-0.135384,-0.296558,NaN,NaN,2.3876
2,2023-01-03,0.0000,0.197442,-0.157954,-0.454512,0.0000,NaN,2.3876
3,2023-01-04,2.3876,0.103623,2.304702,1.850190,0.0000,0.0000,2.5400
4,2023-01-05,0.0000,0.175670,-0.140536,1.709653,2.3876,2.3876,35.1536
5,2023-01-06,2.5400,0.109968,2.452025,4.161679,2.3876,2.3876,38.2524
6,2023-01-07,32.6136,0.020415,32.597268,36.758946,2.5400,4.9276,21.4122
7,2023-01-08,5.6388,0.067273,5.584981,42.505102,35.1536,35.1536,24.2570
8,2023-01-09,15.7734,0.078499,15.710601,58.351087,38.2524,40.7924,8.5090
9,2023-01-10,8.4836,0.068157,8.429074,66.938115,21.4122,54.0258,6.8072


In [6]:
import numpy as np

cond_golden = (
    (df_unit['last_2_days']<3)&
    (df_unit['sum_7_days_SMI'].between(5,12))&
    (df_unit['ET'].between(0.15,0.35))&
    (df_unit['next_2_days'].between(8,35))
)

cond_good = (
    (df_unit['last_2_days']<3)&
    (df_unit['sum_7_days_SMI'].between(0,12))&
    (df_unit['ET'].between(0.15,0.35))&
    (df_unit['next_2_days'].between(8,35))
)

cond_risky = (
    (df_unit['last_2_days']<5)&
    (df_unit['sum_7_days_SMI'].between(-5,2))&
    (df_unit['next_2_days']<35)
)

cond_avoid = (
    (df_unit['last_3_days']>20)|
    (df_unit['sum_7_days_SMI']<-5)|
    (df_unit['sum_7_days_SMI']>15)|
    (df_unit['next_2_days']>40)
)

conditions = [cond_avoid, cond_golden, cond_good, cond_risky]
choices =['Avoid','Golden','Good','Risky']
df_unit['daily_recommendation'] = np.select(conditions, choices, default='Normal')
display(df_unit[['date','rainfall_mm','last_2_days','last_3_days','sum_7_days_SMI','ET','next_2_days','daily_recommendation']].head(1000))

,date,rainfall_mm,last_2_days,last_3_days,sum_7_days_SMI,ET,next_2_days,daily_recommendation
0,2023-01-01,0.0000,NaN,NaN,-0.161174,0.201468,0.0000,Normal
1,2023-01-02,0.0000,NaN,NaN,-0.296558,0.169230,2.3876,Normal
2,2023-01-03,0.0000,0.0000,NaN,-0.454512,0.197442,2.3876,Risky
3,2023-01-04,2.3876,0.0000,0.0000,1.850190,0.103623,2.5400,Risky
4,2023-01-05,0.0000,2.3876,2.3876,1.709653,0.175670,35.1536,Normal
...,...,...,...,...,...,...,...,...
995,2025-09-22,0.0000,1.1176,1.9304,9.453960,0.193491,0.9144,Normal
996,2025-09-23,0.5080,0.0000,1.1176,9.710644,0.197672,2.6416,Normal
997,2025-09-24,0.4064,0.5080,0.5080,8.594841,0.272793,4.1402,Normal
998,2025-09-25,2.2352,0.9144,0.9144,3.847244,0.272524,8.4836,Good


In [12]:
df_unit['year'] = df_unit['date'].dt.year
yearly_status_count = df_unit.groupby(['year','daily_recommendation']).size().unstack(fill_value=0)
yearly_status_count['total'] = yearly_status_count.sum(axis=1)
display(yearly_status_count)

daily_recommendation,Avoid,Golden,Good,Normal,Risky,total
year,,,,,,
2023,254,5,6,66,34,365
2024,246,1,4,35,80,366
2025,260,2,6,63,34,365
2026,192,6,2,65,39,304


In [ ]:
print(df_unit[['last_2_days','last_3_days','sum_7_days_SMI','ET','next_2_days']].describe())

       last_2_days  last_3_days  sum_7_days_SMI           ET  next_2_days
count  1398.000000  1397.000000     1400.000000  1400.000000  1398.000000
mean     12.084622    18.129727       41.249712     0.153227    12.086784
std      19.201178    26.062028       48.823638     0.060547    19.199987
min       0.000000     0.000000       -1.648272    -0.007878     0.000000
25%       0.863600     2.032000       10.305974     0.111536     0.863600
50%       6.121400    10.693400       31.843832     0.147977     6.121400
75%      16.687800    26.263600       60.696034     0.191240    16.687800
max     309.143400   357.352600      559.440916     0.382033   309.143400


In [ ]:
idx = 2
print("Date:", df_unit.loc[idx, 'date'])
print("last_2_days (< 5):", df_unit.loc[idx, 'last_2_days'] < 5)
print("sum_7_days_SMI (between -5 and 2):", -5 <= df_unit.loc[idx, 'sum_7_days_SMI'] <= 2)
print("next_2_days (< 35):", df_unit.loc[idx, 'next_2_days'] < 35)
test_val = df_unit.loc[idx, 'sum_7_days_SMI']
print(" ค่า sum_7_days_SMI จริงคือ:", test_val)

Date: 2023-01-03 00:00:00
last_2_days (< 5): True
sum_7_days_SMI (between -5 and 2): True
next_2_days (< 35): True
 ค่า sum_7_days_SMI จริงคือ: -0.45451184000000006
